# Pickup Distance Calibration

Place the robot at the exact position where the arm can pick up a bottle cap.
Run the cells below to find the bounding box width at that distance.
That width value becomes your PICKUP_THRESHOLD in the main autonomous script.

In [ ]:
import cv2
import json
import base64
from urllib import request
from jetbot import Camera, bgr8_to_jpeg

print("Libraries loaded.")

In [ ]:
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"

# Lower confidence so it picks up even weak detections during calibration
CONFIDENCE_THRESHOLD = 0.5

camera = Camera.instance(width=224, height=224)
print("Camera ready.")

In [ ]:
def get_cap_info():
    frame = camera.value.copy()

    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        print("Failed to encode frame")
        return

    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")

    payload = {
        "api_key": API_KEY,
        "inputs": {
            "image": {"type": "base64", "value": image_b64},
            "confidence": CONFIDENCE_THRESHOLD
        }
    }

    data = json.dumps(payload).encode("utf-8")
    req = request.Request(
        url=API_URL,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "python-urllib/3.6"
        },
        method="POST"
    )

    with request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read().decode("utf-8"))

    predictions = result["outputs"][0]["predictions"]["predictions"]

    if predictions:
        print(f"Found {len(predictions)} cap(s)\n")
        for i, pred in enumerate(predictions):
            print(f"--- Cap {i+1} ---")
            print(f"confidence : {pred['confidence']:.2f}")
            print(f"x (center) : {pred['x']} px  (image center = 112px)")
            print(f"y (center) : {pred['y']} px")
            print(f"width      : {pred['width']} px  <-- use this as PICKUP_THRESHOLD")
            print(f"height     : {pred['height']} px")
            print()
    else:
        print("No cap detected.")
        print("Try lowering CONFIDENCE_THRESHOLD or move the robot closer.")

print("Function ready.")

### Instructions
1. Place the robot at the exact pickup position where the arm can reach the cap
2. Run the cell below
3. Note the **width** value printed
4. Run it 3-4 times to confirm the value is consistent
5. Use that width as your PICKUP_THRESHOLD in the main script

In [ ]:
# Run this once per test position
get_cap_info()

In [ ]:
# Run multiple times to check consistency
import time

NUM_SAMPLES = 5

for i in range(NUM_SAMPLES):
    print(f"=== Sample {i+1} ===")
    get_cap_info()
    time.sleep(2)

In [ ]:
camera.stop()
print("Camera stopped.")